# AI와 함께 타이타닉 데이터 분석 한 사이클 완주하기

데이터 → 전처리 → 시각화 → EDA → Feature → 분류 → 평가 → 모델 선택 → 저장 → Streamlit

> 코드는 AI의 도움을 받아 최소한으로 작성하지만, 분석을 단계별로 진행하고 실제 결과를 보고 다음 행동을 결정하는 사람은 학생입니다.

**사용 원칙**: 실행 계획 → 코드 실행 → 실행 결과 요약 → 실행 결과 분석 순서를 지킵니다. 관찰/해석/가설/한계를 구분하고, AI가 실제 실행 결과를 만들어내게 하지 않습니다.


## STEP 00. 전체 분석 지도

### 실행 계획
전체 흐름과 AI/학생 역할을 확인한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


## STEP 01. 실행 환경 확인

### 실행 계획
현재 Notebook을 실행하는 Python 경로·버전·작업 경로와 주요 패키지 버전을 확인한다.

VS Code에서 선택한 **Python Interpreter**는 터미널에서 사용할 환경이고, Notebook 오른쪽 위의 **Kernel**은 셀을 실행할 환경이다. 둘이 다를 수 있으므로 Kernel에서도 사용할 Python 환경을 선택하고, 아래 `sys.executable` 출력으로 확인한다.

두 셀을 위에서부터 직접 실행한다. 패키지 import 오류가 나면 오류에 나온 패키지와 선택한 Kernel을 확인한다. 필요한 설치는 README 안내에 따라 같은 환경의 터미널에서 직접 수행한 뒤 Kernel을 다시 시작한다. 이 Notebook은 패키지를 자동 설치하거나 오류를 숨기지 않는다.


In [1]:
# 현재 셀을 실행하는 Python과 작업 경로
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Working directory:", Path.cwd())


Python executable: C:\dev\ai-data-analysis\.venv\Scripts\python.exe
Python version: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Working directory: C:\dev\llm-data-analysis-course\notebooks


In [2]:
# 패키지 버전: 오류가 나면 선택한 Kernel의 환경을 확인하세요.
import pandas as pd
import numpy as np
import sklearn
from IPython.display import display

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)


pandas: 3.0.5
numpy: 2.5.1
scikit-learn: 1.9.0


### 실행 결과 요약 — 학생 작성
- 현재 Python:
- 현재 Kernel:
- 주요 패키지:

### 실행 결과 분석 — 학생 작성
- 관찰:
- 해석:
- 가설 / 추가 확인:
- 한계:


## STEP 02. Titanic 데이터 준비와 로딩

### 실행 계획
저장소 루트의 터미널에서 Notebook Kernel과 같은 Python 환경으로 다음 명령을 먼저 실행한다.

```powershell
python scripts/prepare_titanic_data.py
```

준비 스크립트는 공개 원본을 다운로드하고 강의 데이터 기준으로 검증한다. 아래 helper는 **data 폴더만 기준으로** 현재 위치부터 상위 폴더를 탐색하므로 저장소 루트와 `notebooks/`에서 모두 사용할 수 있다.

경로를 확인한 뒤 CSV를 읽고 앞부분·크기·컬럼을 직접 확인한다. `df`는 이후 전체 실습의 원본 기준 DataFrame이며 이 단계에서는 전처리하지 않는다.


In [3]:
from pathlib import Path

def get_project_root(start_path: Path | None = None) -> Path:
    """현재 위치에서 상위 폴더를 탐색해 data 폴더가 있는 프로젝트 루트를 찾는다."""
    current = (start_path or Path.cwd()).resolve()

    for path in (current, *current.parents):
        if (path / "data").is_dir():
            return path

    raise FileNotFoundError(
        "data 폴더가 있는 프로젝트 루트를 찾을 수 없습니다."
    )

project_root = get_project_root()
data_path = project_root / "data" / "titanic" / "train.csv"

if not data_path.is_file():
    raise FileNotFoundError(
        f"Titanic 데이터가 없습니다: {data_path}\n"
        f"저장소 루트({project_root})에서 Kernel과 같은 Python 환경으로 "
        "python scripts/prepare_titanic_data.py 를 실행한 뒤 다시 실행하세요."
    )

print("Data path:", data_path)


Data path: C:\dev\llm-data-analysis-course\data\titanic\train.csv


In [4]:
df = pd.read_csv(data_path)

display(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Shape: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


### 실행 결과 요약 — 학생 작성
- 읽은 파일 경로:
- 행/열 수:
- 컬럼과 앞부분에서 확인한 내용:

### 실행 결과 분석 — 학생 작성
- 관찰:
- 해석:
- 가설 / 추가 확인:
- 한계:


## STEP 03. 데이터 구조와 품질의 첫인상

### 실행 계획
원본 `df`를 그대로 사용해 네 개의 작은 셀을 순서대로 실행한다. 각 출력에서 직접 확인한 사실을 적고, 전처리 방법은 아직 확정하지 않는다.

A: 행/열·컬럼·앞부분·dtype → B: 결측치와 중복 → C: 주요 범주형 값 → D: 기초 통계.

숫자로 저장된 식별자·범주 코드의 평균을 연속형 측정값처럼 해석하지 않도록 주의한다. 통계는 결측값을 제외해 계산되므로 `count`도 함께 살펴본다.


In [5]:
# A. 데이터 구조와 dtype
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
df.info()


Shape: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.7 KB


In [6]:
# B. 결측치 비율은 백분율(%)입니다.
missing_count = df.isna().sum()
missing_rate = df.isna().mean().mul(100)
missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_rate": missing_rate.round(2),
})
display(missing_summary)
print("Duplicate rows:", df.duplicated().sum())


,missing_count,missing_rate
PassengerId,0,0.00
Survived,0,0.00
Pclass,0,0.00
Name,0,0.00
Sex,0,0.00
Age,177,19.87
SibSp,0,0.00
Parch,0,0.00
Ticket,0,0.00
Fare,0,0.00


Duplicate rows: 0


In [7]:
# C. 숫자로 저장된 Target/등급도 범주별로 관찰합니다.
for column in ["Survived", "Pclass", "Sex", "Embarked"]:
    print(f"Category: {column} (결측값 포함)")
    display(df[column].value_counts(dropna=False).rename("count").to_frame())


Category: Survived (결측값 포함)


,count
Survived,
0,549
1,342


Category: Pclass (결측값 포함)


,count
Pclass,
3,491
1,216
2,184


Category: Sex (결측값 포함)


,count
Sex,
male,577
female,314


Category: Embarked (결측값 포함)


,count
Embarked,
S,644
C,168
Q,77
NaN,2


In [8]:
# D. 숫자형과 문자형을 나누어 요약합니다. 원본 df는 바뀌지 않습니다.
print("Numeric summary:")
display(df.describe())
print("Text summary:")
display(df.select_dtypes(include=["object", "string"]).describe())


Numeric summary:


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Text summary:


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


### 출력에서 생각해 볼 질문
- Age 결측치는 어떻게 할 것인가?
- Cabin은 결측치가 많아 보이는가?
- Embarked의 결측치는 많지 않은가?
- 숫자처럼 보여도 실제 의미는 범주형인 컬럼이 있는가?
- PassengerId는 예측 Feature로 의미가 있는가?
- Name, Ticket, Cabin은 바로 버려야 하는가?

아직 결측치 대체·행 삭제·컬럼 제거를 실행하지 않는다. `df_work`는 STEP 05, `df_encoded`는 STEP 07, `model_source`는 STEP 11에서 생성한다.


### 실행 결과 요약 — 학생 작성
- 데이터 구조:
- 결측치와 중복:
- 범주와 기초 통계에서 확인한 내용:

### 실행 결과 분석 — 학생 작성
- 관찰:
- 해석:
- 가설 / 추가 확인:
- 한계:


### 첫 번째 개인 판단 — 학생이 선택
실제 출력을 보고 **추가로 확인하고 싶은 데이터 품질 항목 하나**를 선택한다. 예: 이상치, 범주별 개수, 중복 PassengerId, Fare 분포, Age 범위, 특정 컬럼 unique 개수. AI가 대신 확정하지 않는다.

필요하면 실제 출력과 함께 다음 Prompt로 후보만 요청한다.

> 현재 STEP 03 실행 결과를 기준으로 추가로 확인할 가치가 있는 데이터 품질 분석 후보 3개를 제안해 주세요. 각 후보에 대해 1. 무엇을 확인하는지 2. 왜 필요한지 3. 결과에 따라 다음 판단이 어떻게 달라지는지를 설명해 주세요. 아직 코드는 작성하지 마세요.

- 내가 선택한 항목:
- 선택 이유:
- 진행하지 않은 후보와 이유:
- 추가 확인 후 기록할 실제 결과:
- 다음 판단:


## STEP 04. 분석 문제와 Target 정의

### 실행 계획
원본 `df`에서 Survived의 존재·결측·고유값·class 개수와 비율을 확인하고, 내가 정의한 분석 문제를 한 문장으로 적는다.

### 핵심 개념
Target은 맞혀야 하는 정답이고 Feature는 예측 시점에 사용할 수 있는 입력 정보다. 이 실습의 Target `Survived`에서 0은 생존하지 못함, 1은 생존을 뜻한다. 두 범주를 구분하므로 **Binary Classification(이진 분류)** 문제다. 연속적인 수치를 맞히는 Regression(회귀)과 구분한다.

class별 개수와 비율이 얼마나 다른지 실제 출력으로 판단한다. 불균형의 영향은 이후 분할·평가에서 검토하며 지금 균형을 맞추지는 않는다.

**Target leakage**는 정답을 미리 보는 정보가 입력에 섞이는 것이다. `Survived`를 Feature에 그대로 넣거나 생존·구조 이후에 알게 되는 정보를 쓰는 경우가 예다. 현재 데이터에 그런 추가 컬럼이 있다고 가정하는 것은 아니다. Feature 후보의 의미와 예측 시점부터 검토하고 X/y는 아직 확정하지 않는다.


In [9]:
# Target이 실제로 있는지 먼저 확인합니다.
print("Survived exists:", "Survived" in df.columns)
if "Survived" not in df.columns:
    raise KeyError("Target Survived가 없습니다. STEP 02의 데이터 경로와 컬럼을 확인하세요.")
print("dtype:", df["Survived"].dtype)
print("missing:", df["Survived"].isna().sum())
print("unique:", df["Survived"].unique())


Survived exists: True
dtype: int64
missing: 0
unique: [0 1]


In [10]:
target_summary = pd.DataFrame({
    "count": df["Survived"].value_counts(dropna=False),
    "ratio": df["Survived"].value_counts(normalize=True, dropna=False),
}).sort_index()
display(target_summary)
# ratio는 0~1 비율입니다. class 분포 차이를 직접 비교하세요.


,count,ratio
Survived,,
0,549,0.616162
1,342,0.383838


### 실행 결과 요약 — 학생 작성
- Target 고유값과 결측 여부:
- class별 개수와 비율:
- class 불균형에 대한 내 판단:

### 실행 결과 분석 — 학생 작성
- 관찰:
- 해석:
- 가설 / 추가 확인:
- 한계:


### 개인 판단 — 내가 정의한 분석 문제
- 한 문장으로 적은 분석 문제:
- 이진 분류라고 판단한 이유:
- Feature 후보와 Target의 차이:
- 예측 시점에 사용할 수 있는지 추가 확인할 정보:

내 문장과 실제 Target 출력을 AI에게 제공하고 검토만 요청할 수 있다.

> 내 문제 정의를 대신 작성하지 말고, Target·Feature 혼동과 예측 시점에 따른 누수 위험을 점검해 주세요. 현재 STEP의 실제 실행 결과만 기준으로 답하세요. 관찰·해석·가설·한계를 구분하세요. 추가 분석은 후보만 제시하고, 내가 선택하기 전에는 다음 코드를 작성하거나 다음 STEP으로 진행하지 마세요.


## STEP 05. 결측치 처리

### 실행 계획
원본 `df`에서 탐색용 `df_work`를 이번 STEP에서 한 번만 만든다. 결측 개수·비율 확인 → 원인/규모 생각 → 후보 비교 → 탐색용 처리 → 처리 후 확인 순서로 진행한다. 결측 원인은 출력만으로 단정하지 않는다.

### 핵심 개념과 기본안
아래 기본안은 수업을 위에서 아래로 실행하기 위한 **탐색/EDA용 예시**이며 유일한 정답이나 학생 개인의 결정이 아니다. 먼저 현황과 후보를 보고 최소 한 컬럼의 선택 이유를 직접 기록한다. 수정 없이 실행하면 기본안이 적용되고, 다른 방식을 선택하면 해당 처리 셀을 바꾼 뒤 Kernel을 재시작해 처음부터 실행한다. STEP 06에서 작업본을 새로 만들지 않는다.

| 컬럼 | 탐색용 기본안 | 비교할 후보와 주의점 |
|---|---|---|
| Age | 중앙값 대체 | 평균은 이상값 영향을 받을 수 있고, 중앙값도 분포와 변동성을 단순화한다. 보류/그룹별 대체도 비교한다. |
| Embarked | 최빈값 대체 | 대표 범주가 더 많아진다. Unknown 범주나 보류는 결측의 의미를 다르게 다룬다. |
| Cabin | 결측 상태와 원문 보존 | 결측 비율을 확인하고 단순 fill이나 즉시 drop 대신 Deck/CabinKnown 파생 가능성을 이후 검토한다. |

탐색 단계에서는 전체 `df_work`의 median/mode로 패턴을 보기 쉽게 할 수 있다. **이 값과 처리 결과를 최종 모델 preprocessing에 재사용하면 안 된다.** STEP 11에서 원본을 기준으로 train/test split을 먼저 하고, STEP 12 Pipeline의 `SimpleImputer`가 train 데이터에만 fit되도록 다시 구성한다. test에서는 train에서 배운 기준만 적용한다.


In [11]:
# STEP 05~10에서 이어서 사용할 작업본을 여기서 한 번 생성합니다.
df_work = df.copy()
missing_before = pd.DataFrame({
    "count": df_work.isna().sum(),
    "percentage": df_work.isna().mean().mul(100).round(2),
})
print("Raw / working shape:", df.shape, df_work.shape)
print("Different objects:", df is not df_work)
display(missing_before)
display(missing_before.loc[["Age", "Cabin", "Embarked"]])


Raw / working shape: (891, 12) (891, 12)
Different objects: True


,count,percentage
PassengerId,0,0.00
Survived,0,0.00
Pclass,0,0.00
Name,0,0.00
Sex,0,0.00
Age,177,19.87
SibSp,0,0.00
Parch,0,0.00
Ticket,0,0.00
Fare,0,0.00


,count,percentage
Age,177,19.87
Cabin,687,77.10
Embarked,2,0.22


### 개인 판단 — 처리 전에 후보 비교
Age 처리, Embarked 처리, Cabin 보존/파생/제외 중 최소 하나를 직접 판단한다. 아래는 학생 기록란이며 기본안 적용이 자동으로 개인 선택을 뜻하지 않는다.

| 컬럼 | 나의 선택 | 선택 이유 | 다른 후보를 선택하지 않은 이유 | 최종 모델링에서 다시 확인할 점 |
|---|---|---|---|---|
| Age | | | | |
| Cabin | | | | |
| Embarked | | | | |

> 현재 결측치 결과를 기준으로 Age, Cabin, Embarked 각각에 대해 가능한 처리 방법을 2~3개씩 제안해 주세요. 각 방법의 장점·단점·탐색 분석 영향·최종 모델링 주의점을 설명하세요. 아직 코드는 작성하지 마세요. 현재 STEP의 실제 실행 결과만 기준으로 답하세요. 관찰·해석·가설·한계를 구분하세요. 추가 분석은 후보만 제시하고, 내가 선택하기 전에는 다음 코드를 작성하거나 다음 STEP으로 진행하지 마세요.


In [12]:
# 탐색용 기본 실행안: 학생 선택에 따라 이 셀의 처리 방법을 변경할 수 있습니다.
age_median_eda = df_work["Age"].median()
embarked_mode_eda = df_work["Embarked"].mode().iloc[0]
print("Exploratory Age median:", age_median_eda)
print("Exploratory Embarked mode:", embarked_mode_eda)

df_work["Age"] = df_work["Age"].fillna(age_median_eda)
df_work["Embarked"] = df_work["Embarked"].fillna(embarked_mode_eda)
# Cabin은 원문과 결측 상태를 그대로 보존합니다.


Exploratory Age median: 28.0
Exploratory Embarked mode: S


In [13]:
missing_after = pd.DataFrame({
    "count": df_work.isna().sum(),
    "percentage": df_work.isna().mean().mul(100).round(2),
})
display(pd.concat({"before": missing_before, "after": missing_after}, axis=1))
print("Raw / working shape:", df.shape, df_work.shape)
print("Raw missing counts:")
display(df[["Age", "Cabin", "Embarked"]].isna().sum().to_frame("count"))


before            after           
             count percentage count percentage
PassengerId      0       0.00     0        0.0
Survived         0       0.00     0        0.0
Pclass           0       0.00     0        0.0
Name             0       0.00     0        0.0
Sex              0       0.00     0        0.0
Age            177      19.87     0        0.0
SibSp            0       0.00     0        0.0
Parch            0       0.00     0        0.0
Ticket           0       0.00     0        0.0
Fare             0       0.00     0        0.0
Cabin          687      77.10   687       77.1
Embarked         2       0.22     0        0.0

Raw / working shape: (891, 12) (891, 12)
Raw missing counts:


,count
Age,177
Cabin,687
Embarked,2


### 실행 결과 요약 — 학생 작성
- 처리 전후 결측 개수·비율:
- 행/열 수 변화:
- 기본안 적용 또는 내가 변경한 처리와 이유:
- 원본과 작업본의 차이:

### 실행 결과 분석 — 학생 작성
- 관찰:
- 해석:
- 가설 / 추가 확인:
- 한계:


## STEP 06. 불필요한 컬럼 검토

### 실행 계획
STEP 05의 기존 `df_work`를 이어서 사용한다. 모든 컬럼의 상태를 확인하고 특히 PassengerId·Name·Ticket·Cabin을 검토한다. 컬럼을 삭제하거나 최종 모델 입력을 확정하지 않는다.

### 핵심 개념
| 정책 | 의미 |
|---|---|
| KEEP | 현재 형태를 유지하며 활용 가능성 검토 |
| EXCLUDE | 모델 직접 입력에서 제외할 후보로 기록 |
| DERIVE | 파생 Feature 재료로 검토 |
| DEFER | 근거가 부족해 판단 보류 |

PassengerId는 일반적인 예측 Feature 가치가 낮을 수 있는 식별자지만 행 추적에는 유용하다. Name은 Title, Cabin은 Deck/CabinKnown, Ticket은 prefix/group 관련 Feature 후보의 재료가 될 수 있다. 이는 후보 설명이며 실제 파생이나 최종 선택은 이후에 한다. 특히 Ticket 빈도/그룹처럼 데이터에서 기준을 배우는 변환은 최종 평가에서 train 범위 원칙도 검토해야 한다.

**모델 입력 제외 후보와 데이터에서 삭제하는 것은 다르다.** Name·Ticket·Cabin과 원본 reference를 보존한다. Survived는 Target이므로 Feature로 사용하지 않는다.


In [14]:
# 모든 컬럼의 역할을 검토할 근거를 봅니다. df_work를 다시 만들지 않습니다.
column_review = pd.DataFrame({
    "dtype": df_work.dtypes.astype(str),
    "unique_count": df_work.nunique(dropna=True),
    "missing_count": df_work.isna().sum(),
    "missing_percentage": df_work.isna().mean().mul(100).round(2),
})
display(column_review)


,dtype,unique_count,missing_count,missing_percentage
PassengerId,int64,891,0,0.0
Survived,int64,2,0,0.0
Pclass,int64,3,0,0.0
Name,str,891,0,0.0
Sex,str,2,0,0.0
Age,float64,88,0,0.0
SibSp,int64,7,0,0.0
Parch,int64,7,0,0.0
Ticket,str,681,0,0.0
Fare,float64,248,0,0.0


In [15]:
review_candidates = ["PassengerId", "Name", "Ticket", "Cabin"]
existing_review_cols = [col for col in review_candidates if col in df_work.columns]
print("Existing review columns:", existing_review_cols)
for col in existing_review_cols:
    print(col, "examples:", df_work[col].dropna().head(5).tolist())
print("Columns preserved:", df_work.columns.tolist())


Existing review columns: ['PassengerId', 'Name', 'Ticket', 'Cabin']
PassengerId examples: [1, 2, 3, 4, 5]
Name examples: ['Braund, Mr. Owen Harris', 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)', 'Heikkinen, Miss Laina', 'Futrelle, Mrs. Jacques Heath (Lily May Peel)', 'Allen, Mr. William Henry']
Ticket examples: ['A/5 21171', 'PC 17599', 'STON/O2. 3101282', '113803', '373450']
Cabin examples: ['C85', 'C123', 'E46', 'G6', 'C103']
Columns preserved: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


### 실행 결과 요약 — 학생 작성
- 컬럼별 고유값·결측·대표값에서 확인한 사실:
- 바로 사용할지 판단하기 어려운 이유:

### 실행 결과 분석 — 학생 작성
- 관찰:
- 해석:
- 가설 / 추가 확인:
- 한계:


### 개인 판단 — 컬럼 정책
최소 한 컬럼에 KEEP / EXCLUDE / DERIVE / DEFER 중 하나와 이유를 직접 기록한다. 나머지 컬럼도 위 표를 보고 추가할 수 있다. 코드의 기본 실행은 모든 컬럼을 보존하며 학생 선택을 자동 확정하지 않는다.

| Column | Decision | Reason | 이후 확인할 내용 |
|---|---|---|---|
| PassengerId | | | |
| Name | | | |
| Ticket | | | |
| Cabin | | | |

> 현재 컬럼 상태를 기준으로 KEEP / EXCLUDE / DERIVE / DEFER 후보와 장단점만 설명해 주세요. 최종 판단을 대신하거나 컬럼을 삭제하지 마세요. 현재 STEP의 실제 실행 결과만 기준으로 답하세요. 관찰·해석·가설·한계를 구분하세요. 추가 분석은 후보만 제시하고, 내가 선택하기 전에는 다음 코드를 작성하거나 다음 STEP으로 진행하지 마세요.


## STEP 07. 범주형 데이터 변환 원리

### 실행 계획
기존 `df_work`에서 Sex·Embarked·Pclass의 실제 값을 확인한 뒤 교육용 `df_encoded`로 한 번 복사한다. 이 복사본에서 mapping과 One-Hot을 실행하고 행 수·Target·변경 컬럼을 확인한다. 이후 STEP 08~10의 EDA는 계속 `df_work`를 사용한다.

### 핵심 개념과 기본 실행안
Label/Mapping은 각 범주에 숫자를 대응시키는 표현이다. Sex 같은 이진 범주는 두 표시값으로 표현할 수 있으며 숫자는 가치나 우열을 뜻하지 않는다. Embarked 같은 명목형 항구에 0/1/2를 부여하면 일부 모델이 없는 순서/거리 관계를 읽을 수 있다. One-Hot Encoding은 범주별 표시 컬럼을 만든다.

Pclass는 숫자로 저장되어도 등급이라는 범주적 의미와 순서가 있다. 숫자 그대로 쓰는지 범주형으로 다루는지는 학생이 판단한다.

기본안은 Sex의 female→0 / male→1 mapping, Embarked One-Hot, Pclass 현재 값 유지다. 원리 비교용 예시이지 모든 학생의 정답이 아니다. `drop_first=False`로 모든 범주 컬럼을 보여 준다. 학생은 실제 출력과 아래 판단란을 보고 전략을 바꿀 수 있으며, 변경 시 Kernel을 재시작하고 처음부터 실행한다.


In [16]:
# 인코딩 전에 사람이 읽을 수 있는 실제 범주를 확인합니다.
for col in ["Sex", "Embarked", "Pclass"]:
    print(col, "dtype:", df_work[col].dtype)
    print("unique:", df_work[col].unique())
    display(df_work[col].value_counts(dropna=False).rename("count").to_frame())


Sex dtype: str
unique: <ArrowStringArray>
['male', 'female']
Length: 2, dtype: str


,count
Sex,
male,577
female,314


Embarked dtype: str
unique: <ArrowStringArray>
['S', 'C', 'Q']
Length: 3, dtype: str


,count
Embarked,
S,646
C,168
Q,77


Pclass dtype: int64
unique: [3 1 2]


,count
Pclass,
3,491
1,216
2,184


### 개인 판단 — 인코딩 전략
- Sex는 mapping이 적절한가?
- Embarked는 One-Hot이 적절한가?
- Pclass는 숫자형 그대로 볼 것인가, 범주형으로 볼 것인가?

| 컬럼 | 내가 선택한 방식 | 이유 | 다른 방식의 장단점 |
|---|---|---|---|
| Sex | | | |
| Embarked | | | |
| Pclass | | | |

> 실제 범주 출력을 기준으로 mapping과 One-Hot의 장단점을 비교하고 Pclass의 의미도 검토해 주세요. 선택은 내가 하며 아직 코드는 작성하지 마세요. 현재 STEP의 실제 실행 결과만 기준으로 답하세요. 관찰·해석·가설·한계를 구분하세요. 추가 분석은 후보만 제시하고, 내가 선택하기 전에는 다음 코드를 작성하거나 다음 STEP으로 진행하지 마세요.


In [17]:
# 교육용 복사본 생성은 이곳에서 한 번만 합니다.
df_encoded = df_work.copy()
encoding_columns_before = df_encoded.columns.tolist()
encoding_rows_before = len(df_encoded)

sex_mapping = {"female": 0, "male": 1}
if not df_encoded["Sex"].isin(sex_mapping).all():
    raise ValueError("Sex에 결측 또는 mapping에 없는 범주가 있습니다. 실제 값을 다시 확인하세요.")
df_encoded["Sex"] = df_encoded["Sex"].map(sex_mapping)
display(pd.DataFrame({"before": df_work["Sex"], "after": df_encoded["Sex"]}).head())


,before,after
0,male,1
1,female,0
2,female,0
3,female,0
4,male,1


In [18]:
# 교육용 One-Hot: df_work에는 적용하지 않습니다.
df_encoded = pd.get_dummies(
    df_encoded, columns=["Embarked"], drop_first=False, dtype=int,
)
encoded_port_cols = [col for col in df_encoded.columns if col.startswith("Embarked_")]
print("Before columns:", encoding_columns_before)
print("After columns:", df_encoded.columns.tolist())
display(pd.concat([df_work[["Embarked"]], df_encoded[encoded_port_cols]], axis=1).head())


Before columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
After columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked_C', 'Embarked_Q', 'Embarked_S']


,Embarked,Embarked_C,Embarked_Q,Embarked_S
0,S,0,0,1
1,C,1,0,0
2,S,0,0,1
3,S,0,0,1
4,S,0,0,1


In [19]:
print("Rows preserved:", encoding_rows_before == len(df_encoded))
print("Target preserved:", df_encoded["Survived"].equals(df_work["Survived"]))
print("df_work category dtypes:")
display(df_work[["Sex", "Embarked", "Pclass"]].dtypes.to_frame("dtype"))
print("Separate objects:", df_encoded is not df_work)
# 위 출력과 함께 df_work의 문자열 범주가 유지되는지 확인합니다.
display(df_work[["Sex", "Embarked", "Pclass"]].head())


Rows preserved: True
Target preserved: True
df_work category dtypes:


,dtype
Sex,str
Embarked,str
Pclass,int64


Separate objects: True


,Sex,Embarked,Pclass
0,male,S,3
1,female,C,1
2,female,S,3
3,female,S,1
4,male,S,3


### 최종 Pipeline과의 차이
현재 `pd.get_dummies()` 결과인 **df_encoded는 인코딩 원리 학습용이며 최종 모델 입력이 아니다.** 전체 데이터에서 category vocabulary(범주 목록)를 먼저 정하면 test 정보가 전처리에 섞여 공정한 평가 원칙을 깨뜨릴 수 있다.

최종 모델은 STEP 11에서 원본 기반 입력을 다시 준비해 train/test split을 먼저 한다. STEP 12의 `ColumnTransformer` + `OneHotEncoder(handle_unknown="ignore")` + `Pipeline` 안에서 인코더가 train 데이터에만 fit되고 test에는 transform만 수행한다. `handle_unknown="ignore"`는 학습에 없던 범주가 들어왔을 때 변환 중단을 막는 옵션이다. 여기서는 이 Pipeline을 구현하거나 X/최종 모델 입력을 만들지 않는다.


### 실행 결과 요약 — 학생 작성
- 적용한 기본안 또는 변경 전략:
- mapping 전후 값과 One-Hot 생성 컬럼:
- 행 수·Target·df_work 보존 확인:
- df_encoded를 최종 모델에 직접 사용할 수 없는 이유:

### 실행 결과 분석 — 학생 작성
- 관찰:
- 해석:
- 가설 / 추가 확인:
- 한계:


## STEP 08. 시각화와 기본 패턴 확인

### 실행 계획
`df_work`로 질문 → 그래프 → 관찰 → 해석을 수행한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [20]:
# STEP 08
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 09. 기초 통계와 EDA

### 실행 계획
같은 `df_work`로 비율·표본 수·평균/중앙값·그룹 비교를 수행한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [21]:
# STEP 09
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 10. Feature 설계

### 실행 계획
결정적 파생 Feature와 데이터에서 기준을 학습하는 Feature를 구분한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [22]:
# STEP 10
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 11. 학습/테스트 데이터 준비

### 실행 계획
원본 `df`에서 `model_source`를 다시 만들고 전처리보다 먼저 split한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [23]:
# STEP 11
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 12. Baseline 분류 모델 학습

### 실행 계획
`ColumnTransformer + Pipeline + LogisticRegression`을 `X_train`에만 fit한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [24]:
# STEP 12
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 13. 모델 성능 평가와 오류 분석

### 실행 계획
Accuracy·Confusion Matrix·Precision·Recall·F1·오분류 사례를 확인한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [25]:
# STEP 13
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 14. 추가 모델/알고리즘 선택

### 실행 계획
학생이 후보 하나를 선택하고 같은 전처리 계약의 `additional_pipeline`을 만든다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [26]:
# STEP 14
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 15. 모델 비교와 최종 모델 선정

### 실행 계획
train-only CV를 보강 근거로 사용하고 `final_pipeline`을 확정한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [27]:
# STEP 15
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 16. 최종 Pipeline 저장과 새 승객 예측

### 실행 계획
Pipeline/계약을 저장하고 재로딩 예측 일치를 검증한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [28]:
# STEP 16
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

## STEP 17. Streamlit 예측 앱 구현

### 실행 계획
저장 Pipeline을 그대로 사용해 입력 → 예측 → 확률 표시를 연결한다.

### 실행 결과 요약
- 

### 실행 결과 분석
- 관찰:
- 해석:
- 가설/추가 확인:
- 한계:


In [29]:
# STEP 17
# 상세 실습 가이드를 확인한 뒤, 이 STEP에 필요한 최소 코드만 작성하세요.

# 최종 회고

- 가장 중요한 관찰:
- 내가 직접 내린 판단:
- AI 제안을 수정하거나 거절한 사례:
- 최종 모델과 선택 이유:
- 현재 모델/앱의 한계:
- 다음 확장: 데이터 수집 자동화 → 분석 자동화 → 보고서 자동화 → 사람 승인 기반 Data Analysis Agent
